# Coconut Mite Detection Model v11
## Balanced Training with Proper Evaluation

**Requirements:**
- No data leaking (strict train/val/test split)
- No overfitting (early stopping, regularization)
- Class-wise Precision, Recall, F1-score
- Balanced metrics across all classes
- Accuracy close to F1-score

**Dataset:** `dataset_v4_clean` (3 classes: coconut_mite, healthy, not_coconut)

## 1. Setup and Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Sklearn imports for metrics
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    precision_recall_fscore_support,
    accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration

In [ ]:
# Paths
BASE_DIR = r'D:\SLIIT\Reaserch Project\CoconutHealthMonitor\Research\ml'
DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'pest_mite', 'dataset_v4_clean')
MODEL_DIR = os.path.join(BASE_DIR, 'models', 'coconut_mite_v11')

# Create model directory
os.makedirs(MODEL_DIR, exist_ok=True)

# Training configuration
CONFIG = {
    'img_size': 224,
    'batch_size': 32,
    'epochs': 50,
    'learning_rate': 0.0001,
    'patience': 10,  # Early stopping patience
    'min_delta': 0.001,
    'l2_reg': 0.01,
    'dropout': 0.5,
    'focal_gamma': 2.0,
    'focal_alpha': 0.25,
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Data Exploration

In [ ]:
def count_images(directory):
    """Count images in each class folder"""
    counts = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            counts[class_name] = count
    return counts

# Count images in each split
train_counts = count_images(os.path.join(DATA_DIR, 'train'))
val_counts = count_images(os.path.join(DATA_DIR, 'validation'))
test_counts = count_images(os.path.join(DATA_DIR, 'test'))

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)

# Create summary dataframe
summary_df = pd.DataFrame({
    'Train': train_counts,
    'Validation': val_counts,
    'Test': test_counts
})
summary_df['Total'] = summary_df.sum(axis=1)
summary_df.loc['TOTAL'] = summary_df.sum()

print(summary_df)
print("\n" + "=" * 50)

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

splits = ['Train', 'Validation', 'Test']
counts_list = [train_counts, val_counts, test_counts]
colors = ['#4CAF50', '#2196F3', '#FF9800']

for ax, split, counts, color in zip(axes, splits, counts_list, colors):
    classes = list(counts.keys())
    values = list(counts.values())
    
    bars = ax.bar(classes, values, color=color, alpha=0.8, edgecolor='black')
    ax.set_title(f'{split} Set Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of Images')
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
                str(val), ha='center', va='bottom', fontweight='bold')
    
    ax.set_ylim(0, max(values) * 1.15)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'dataset_distribution.png'), dpi=150)
plt.show()

print("\nClass distribution chart saved!")

## 4. Data Loading (No Data Leaking)

In [ ]:
# Data augmentation ONLY for training set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest',
    brightness_range=[0.8, 1.2]
)

# NO augmentation for validation and test - only rescaling
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("Data generators created:")
print("  - Training: WITH augmentation (rotation, shift, zoom, flip, brightness)")
print("  - Validation: NO augmentation (only rescaling)")
print("  - Test: NO augmentation (only rescaling)")

In [ ]:
# Load datasets
train_generator = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'),
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'validation'),
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=False  # Important for evaluation
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'test'),
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=False  # Important for evaluation
)

# Get class names and indices
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print(f"\nClasses: {class_names}")
print(f"Number of classes: {num_classes}")
print(f"\nClass indices: {train_generator.class_indices}")

## 5. Class Weights Calculation

In [ ]:
# Calculate class weights to handle imbalance
train_labels = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = dict(enumerate(class_weights_array))

print("Class Weights (to handle imbalance):")
print("=" * 40)
for idx, class_name in enumerate(class_names):
    count = train_counts[class_name]
    weight = class_weights[idx]
    print(f"  {class_name}: weight={weight:.4f} (count={count})")

print("\nHigher weight = less samples = model pays more attention")

## 6. Focal Loss Definition

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    """
    Focal Loss for handling class imbalance
    - gamma: focusing parameter (higher = more focus on hard examples)
    - alpha: class weight balancing
    """
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        
        # Cross entropy
        cross_entropy = -y_true * tf.math.log(y_pred)
        
        # Focal weight
        weight = alpha * y_true * tf.pow(1 - y_pred, gamma)
        
        # Focal loss
        focal = weight * cross_entropy
        
        return tf.reduce_mean(tf.reduce_sum(focal, axis=-1))
    
    return focal_loss_fixed

print(f"Focal Loss configured with gamma={CONFIG['focal_gamma']}, alpha={CONFIG['focal_alpha']}")
print("  - Helps model focus on hard-to-classify examples")
print("  - Reduces impact of easy examples")

## 7. Model Architecture

In [ ]:
def create_model(num_classes, l2_reg=0.01, dropout=0.5):
    """Create EfficientNetB0-based model with regularization"""
    
    # Base model (pre-trained on ImageNet)
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(CONFIG['img_size'], CONFIG['img_size'], 3)
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build model
    inputs = keras.Input(shape=(CONFIG['img_size'], CONFIG['img_size'], 3))
    
    # Base model
    x = base_model(inputs, training=False)
    
    # Global pooling
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense layers with regularization
    x = layers.Dense(512, kernel_regularizer=l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)
    
    x = layers.Dense(256, kernel_regularizer=l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)
    
    # Output layer
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    
    return model, base_model

# Create model
model, base_model = create_model(
    num_classes=num_classes,
    l2_reg=CONFIG['l2_reg'],
    dropout=CONFIG['dropout']
)

model.summary()

In [ ]:
# Compile model
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate']),
    loss=focal_loss(gamma=CONFIG['focal_gamma'], alpha=CONFIG['focal_alpha']),
    metrics=['accuracy']
)

print("Model compiled with:")
print(f"  - Optimizer: Adam (lr={CONFIG['learning_rate']})")
print(f"  - Loss: Focal Loss (gamma={CONFIG['focal_gamma']}, alpha={CONFIG['focal_alpha']})")
print(f"  - Metrics: accuracy")

## 8. Callbacks Setup

In [ ]:
# Callbacks for training
callbacks_list = [
    # Early stopping to prevent overfitting
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['patience'],
        min_delta=CONFIG['min_delta'],
        restore_best_weights=True,
        verbose=1
    ),
    
    # Model checkpoint - save best model
    callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'best_model.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    
    # Reduce learning rate when stuck
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured:")
print(f"  - EarlyStopping: patience={CONFIG['patience']}, min_delta={CONFIG['min_delta']}")
print(f"  - ModelCheckpoint: saves best model based on val_loss")
print(f"  - ReduceLROnPlateau: reduces lr when stuck")

## 9. Phase 1: Train with Frozen Base

In [ ]:
print("="*60)
print("PHASE 1: Training with frozen base model")
print("="*60)
print(f"Training for up to {CONFIG['epochs']} epochs...")
print(f"Base model layers: FROZEN")
print()

history_phase1 = model.fit(
    train_generator,
    epochs=CONFIG['epochs'],
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=callbacks_list,
    verbose=1
)

print("\nPhase 1 training complete!")

## 10. Phase 2: Fine-tuning (Unfreeze Base)

In [ ]:
print("="*60)
print("PHASE 2: Fine-tuning with unfrozen base model")
print("="*60)

# Unfreeze top layers of base model
base_model.trainable = True

# Freeze early layers, unfreeze last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum([1 for layer in base_model.layers if layer.trainable])
print(f"Unfrozen {trainable_count} layers in base model for fine-tuning")

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate'] / 10),
    loss=focal_loss(gamma=CONFIG['focal_gamma'], alpha=CONFIG['focal_alpha']),
    metrics=['accuracy']
)

print(f"Learning rate reduced to {CONFIG['learning_rate'] / 10}")
print()

In [ ]:
# Continue training
history_phase2 = model.fit(
    train_generator,
    epochs=30,  # Additional epochs for fine-tuning
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=callbacks_list,
    verbose=1
)

print("\nPhase 2 fine-tuning complete!")

## 11. Training History Visualization

In [ ]:
# Combine histories
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history.keys():
        combined[key] = h1.history[key] + h2.history[key]
    return combined

combined_history = combine_histories(history_phase1, history_phase2)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(combined_history['loss'], label='Training Loss', color='#2196F3', linewidth=2)
axes[0].plot(combined_history['val_loss'], label='Validation Loss', color='#F44336', linewidth=2)
axes[0].axvline(x=len(history_phase1.history['loss']), color='gray', linestyle='--', label='Fine-tuning Start')
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(combined_history['accuracy'], label='Training Accuracy', color='#4CAF50', linewidth=2)
axes[1].plot(combined_history['val_accuracy'], label='Validation Accuracy', color='#FF9800', linewidth=2)
axes[1].axvline(x=len(history_phase1.history['accuracy']), color='gray', linestyle='--', label='Fine-tuning Start')
axes[1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_history.png'), dpi=150)
plt.show()

print(f"\nFinal Training Accuracy: {combined_history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {combined_history['val_accuracy'][-1]:.4f}")

## 12. Load Best Model

In [ ]:
# Load the best model saved by checkpoint
best_model_path = os.path.join(MODEL_DIR, 'best_model.keras')

# Custom objects for loading model with focal loss
custom_objects = {'focal_loss_fixed': focal_loss(CONFIG['focal_gamma'], CONFIG['focal_alpha'])}

best_model = keras.models.load_model(best_model_path, custom_objects=custom_objects)
print(f"Best model loaded from: {best_model_path}")

## 13. Evaluation on Validation Set

In [ ]:
print("="*60)
print("VALIDATION SET EVALUATION")
print("="*60)

# Get predictions on validation set
val_generator.reset()
val_predictions = best_model.predict(val_generator, verbose=1)
val_pred_classes = np.argmax(val_predictions, axis=1)
val_true_classes = val_generator.classes

# Calculate metrics
val_accuracy = accuracy_score(val_true_classes, val_pred_classes)
val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(
    val_true_classes, val_pred_classes, average='weighted'
)

print(f"\n{'Metric':<20} {'Value':<10}")
print("-" * 30)
print(f"{'Accuracy':<20} {val_accuracy:.4f}")
print(f"{'Precision (weighted)':<20} {val_precision:.4f}")
print(f"{'Recall (weighted)':<20} {val_recall:.4f}")
print(f"{'F1-Score (weighted)':<20} {val_f1:.4f}")

In [ ]:
# Class-wise metrics for validation
print("\n" + "="*60)
print("VALIDATION SET - CLASS-WISE METRICS")
print("="*60)

val_report = classification_report(
    val_true_classes, 
    val_pred_classes, 
    target_names=class_names,
    digits=4
)
print(val_report)

## 14. Evaluation on Test Set (Final Evaluation)

In [ ]:
print("="*60)
print("TEST SET EVALUATION (FINAL)")
print("="*60)

# Get predictions on test set
test_generator.reset()
test_predictions = best_model.predict(test_generator, verbose=1)
test_pred_classes = np.argmax(test_predictions, axis=1)
test_true_classes = test_generator.classes

# Calculate metrics
test_accuracy = accuracy_score(test_true_classes, test_pred_classes)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    test_true_classes, test_pred_classes, average='weighted'
)

print(f"\n{'Metric':<20} {'Value':<10}")
print("-" * 30)
print(f"{'Accuracy':<20} {test_accuracy:.4f}")
print(f"{'Precision (weighted)':<20} {test_precision:.4f}")
print(f"{'Recall (weighted)':<20} {test_recall:.4f}")
print(f"{'F1-Score (weighted)':<20} {test_f1:.4f}")

In [ ]:
# Class-wise metrics for test set
print("\n" + "="*60)
print("TEST SET - CLASS-WISE METRICS")
print("="*60)

test_report = classification_report(
    test_true_classes, 
    test_pred_classes, 
    target_names=class_names,
    digits=4,
    output_dict=True
)

# Print formatted report
print(classification_report(
    test_true_classes, 
    test_pred_classes, 
    target_names=class_names,
    digits=4
))

In [ ]:
# Create class-wise metrics comparison chart
fig, ax = plt.subplots(figsize=(12, 6))

metrics_df = pd.DataFrame({
    'Class': class_names,
    'Precision': [test_report[c]['precision'] for c in class_names],
    'Recall': [test_report[c]['recall'] for c in class_names],
    'F1-Score': [test_report[c]['f1-score'] for c in class_names]
})

x = np.arange(len(class_names))
width = 0.25

bars1 = ax.bar(x - width, metrics_df['Precision'], width, label='Precision', color='#2196F3', alpha=0.8)
bars2 = ax.bar(x, metrics_df['Recall'], width, label='Recall', color='#4CAF50', alpha=0.8)
bars3 = ax.bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', color='#FF9800', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('Class-wise Precision, Recall, F1-Score (Test Set)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'class_metrics.png'), dpi=150)
plt.show()

print("\nClass metrics chart saved!")

## 15. Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_true_classes, test_pred_classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# Normalized (percentages)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens', 
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

print("Confusion matrix saved!")

## 16. Metrics Balance Check

In [ ]:
print("="*60)
print("METRICS BALANCE CHECK")
print("="*60)

# Check if metrics are balanced
print("\n1. Overall Metrics Comparison:")
print(f"   Accuracy:  {test_accuracy:.4f}")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   F1-Score:  {test_f1:.4f}")

# Calculate differences
diff_acc_f1 = abs(test_accuracy - test_f1)
diff_prec_rec = abs(test_precision - test_recall)

print(f"\n   |Accuracy - F1|: {diff_acc_f1:.4f} {'(GOOD)' if diff_acc_f1 < 0.05 else '(Needs improvement)'}")
print(f"   |Precision - Recall|: {diff_prec_rec:.4f} {'(GOOD)' if diff_prec_rec < 0.05 else '(Needs improvement)'}")

# Class-wise balance check
print("\n2. Class-wise F1-Score Balance:")
f1_scores = [test_report[c]['f1-score'] for c in class_names]
f1_std = np.std(f1_scores)
f1_range = max(f1_scores) - min(f1_scores)

for c in class_names:
    print(f"   {c}: {test_report[c]['f1-score']:.4f}")

print(f"\n   F1 Std Dev: {f1_std:.4f} {'(GOOD)' if f1_std < 0.1 else '(Needs improvement)'}")
print(f"   F1 Range: {f1_range:.4f} {'(GOOD)' if f1_range < 0.15 else '(Needs improvement)'}")

## 17. Save Model and Info

In [ ]:
# Save model info
model_info = {
    'version': 'v11',
    'model_name': 'coconut_mite_v11',
    'architecture': 'EfficientNetB0',
    'input_shape': [CONFIG['img_size'], CONFIG['img_size'], 3],
    'num_classes': num_classes,
    'class_names': class_names,
    'class_indices': train_generator.class_indices,
    'training_config': CONFIG,
    'metrics': {
        'test_accuracy': float(test_accuracy),
        'test_precision': float(test_precision),
        'test_recall': float(test_recall),
        'test_f1_score': float(test_f1),
        'class_wise': {
            c: {
                'precision': float(test_report[c]['precision']),
                'recall': float(test_report[c]['recall']),
                'f1-score': float(test_report[c]['f1-score']),
                'support': int(test_report[c]['support'])
            } for c in class_names
        }
    },
    'dataset': {
        'train_samples': sum(train_counts.values()),
        'validation_samples': sum(val_counts.values()),
        'test_samples': sum(test_counts.values()),
        'class_distribution': {
            'train': train_counts,
            'validation': val_counts,
            'test': test_counts
        }
    },
    'trained_at': datetime.now().isoformat()
}

# Save model info to JSON
info_path = os.path.join(MODEL_DIR, 'model_info.json')
with open(info_path, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"Model info saved to: {info_path}")

In [ ]:
# List all saved files
print("\n" + "="*60)
print("SAVED FILES")
print("="*60)

for f in os.listdir(MODEL_DIR):
    filepath = os.path.join(MODEL_DIR, f)
    size = os.path.getsize(filepath) / (1024 * 1024)  # MB
    print(f"  {f}: {size:.2f} MB")

## 18. Final Summary

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE - FINAL SUMMARY")
print("="*60)

print(f"\nModel: Coconut Mite Detection v11")
print(f"Architecture: EfficientNetB0 (Transfer Learning)")
print(f"Classes: {class_names}")

print(f"\n{'TEST SET RESULTS':^40}")
print("-"*40)
print(f"{'Accuracy:':<20} {test_accuracy*100:.2f}%")
print(f"{'Precision:':<20} {test_precision*100:.2f}%")
print(f"{'Recall:':<20} {test_recall*100:.2f}%")
print(f"{'F1-Score:':<20} {test_f1*100:.2f}%")

print(f"\n{'CLASS-WISE F1-SCORES':^40}")
print("-"*40)
for c in class_names:
    print(f"{c:<20} {test_report[c]['f1-score']*100:.2f}%")

print(f"\n{'BALANCE CHECK':^40}")
print("-"*40)
balance_ok = diff_acc_f1 < 0.05 and diff_prec_rec < 0.05 and f1_std < 0.1
print(f"Metrics balanced: {'YES' if balance_ok else 'NEEDS IMPROVEMENT'}")

print(f"\nModel saved to: {MODEL_DIR}")
print("="*60)